# Notebook simplificado — Replicação da CNN (COVID-19, Pneumonia, Tuberculose, Normal)

Este notebook faz o pipeline inteiro, do zero: organiza as bases de dados baixadas,
treina a rede e avalia o resultado.

**Pré-requisito:** ter baixado as 3 bases de dados públicas (COVID-19, Pneumonia,
Tuberculose) em pastas separadas no seu computador. Nada além disso precisa existir
ainda — este notebook cria toda a estrutura necessária.

## Bloco 1 — Importar as bibliotecas

Aqui só carregamos as ferramentas que vamos usar no resto do notebook. Nenhuma delas
processa nada ainda, só fica disponível para uso.

- `os`: usada para navegar em pastas e arquivos do computador.
- `tensorflow` / `keras`: biblioteca principal para criar e treinar a rede neural.
- `numpy`: usada para trabalhar com listas de números (arrays) de forma rápida.
- `matplotlib`: usada para desenhar gráficos e imagens na tela.
- `sklearn.metrics`: traz funções prontas para calcular as métricas de avaliação
  (precisão, recall, F1-score, matriz de confusão).

In [7]:
# os: biblioteca embutida do Python para mexer com pastas e arquivos do computador
import os

# TensorFlow: biblioteca principal de deep learning. Keras é a parte "fácil de usar" dela.
from tensorflow import keras
from tensorflow.keras import layers

# numpy: trabalha com listas de números de forma rápida (arrays)
import numpy as np

# matplotlib: desenha gráficos e imagens
import matplotlib.pyplot as plt

# scikit-learn: calcula as métricas prontas (precisão, recall, F1, matriz de confusão)
from sklearn.metrics import classification_report, confusion_matrix


## Bloco 2 — Carregar os caminhos das bases de dados

Os caminhos das 3 bases de dados ficam no arquivo `config.py`, na mesma pasta
deste notebook (veja o README para instruções de como criar esse arquivo).

In [10]:
# Importa os caminhos das 3 bases de dados, definidos no arquivo config.py
from config import RAIZ_COVID, RAIZ_PNEUMONIA, RAIZ_TUBERCULOSE

# Pasta nova que vamos criar, já organizada em 4 classes, pronta para o Keras usar
PASTA_DADOS = "dataset_unificado"

## Bloco 3 — Organizar as imagens em 4 pastas de classe

O Keras só consegue carregar imagens automaticamente se elas estiverem organizadas
assim: uma pasta por classe, cada uma só com as imagens daquela classe.

Como as 3 bases baixadas têm estruturas internas diferentes, este bloco:
1. cria a pasta `dataset_unificado/` com 4 subpastas (uma por classe)
2. para cada classe, copia as imagens da pasta de origem certa para dentro da
   subpasta correspondente

Isso só precisa ser rodado **uma vez** — depois que a pasta `dataset_unificado/`
já existir com as imagens dentro, pode pular direto para o Bloco 4 nas próximas vezes.

In [11]:
# Função que copia todas as imagens de uma pasta de origem para a pasta de destino
def copiar_imagens_da_classe(nome_classe, pastas_de_origem, extensao):

    # Pasta final dessa classe, dentro de dataset_unificado/
    pasta_destino = os.path.join(PASTA_DADOS, nome_classe)

    # Cria a pasta se ela ainda não existir
    os.makedirs(pasta_destino, exist_ok=True)

    contador = 0

    # Pode haver mais de uma pasta de origem (ex: pneumonia tem train/test/val)
    for pasta_origem in pastas_de_origem:

        # Se a pasta não existir, avisa e pula para a próxima
        if not os.path.isdir(pasta_origem):
            print("Aviso: pasta não encontrada:", pasta_origem)
            continue

        # Percorre cada arquivo dentro da pasta de origem
        for nome_arquivo in os.listdir(pasta_origem):

            # Só copia arquivos com a extensão certa (ignora .xlsx, .txt, etc.)
            if nome_arquivo.lower().endswith(extensao):
                caminho_origem = os.path.join(pasta_origem, nome_arquivo)
                caminho_destino = os.path.join(pasta_destino, nome_arquivo)

                # Copia o arquivo de verdade, se ainda não tiver sido copiado antes
                if not os.path.exists(caminho_destino):
                    with open(caminho_origem, "rb") as arquivo_original:
                        with open(caminho_destino, "wb") as arquivo_novo:
                            arquivo_novo.write(arquivo_original.read())
                contador += 1

    print("Classe '" + nome_classe + "':", contador, "imagens")

# --- Organizando cada uma das 4 classes ---

copiar_imagens_da_classe(
    "COVID-19",
    [os.path.join(RAIZ_COVID, "COVID", "images")],
    ".png"
)

copiar_imagens_da_classe(
    "Normal",
    [os.path.join(RAIZ_COVID, "Normal", "images")],
    ".png"
)

copiar_imagens_da_classe(
    "Pneumonia",
    [
        os.path.join(RAIZ_PNEUMONIA, "train", "PNEUMONIA"),
        os.path.join(RAIZ_PNEUMONIA, "test", "PNEUMONIA"),
        os.path.join(RAIZ_PNEUMONIA, "val", "PNEUMONIA"),
    ],
    ".jpeg"
)

copiar_imagens_da_classe(
    "Tuberculosis",
    [os.path.join(RAIZ_TUBERCULOSE, "Tuberculosis")],
    ".png"
)


Classe 'COVID-19': 3616 imagens
Classe 'Normal': 10192 imagens
Classe 'Pneumonia': 4273 imagens
Classe 'Tuberculosis': 700 imagens


## Bloco 4 — Carregar as imagens em treino e teste

Aqui pedimos ao Keras para ler as imagens da pasta `dataset_unificado/` e:
- redimensionar todas para 300x300 pixels (a rede precisa de um tamanho fixo de entrada)
- separar 80% das imagens para treino e 20% para teste
- organizar as imagens em grupos de 32 (chamados de "lote" ou "batch")

A função `image_dataset_from_directory` já identifica sozinha as 4 classes, olhando
o nome de cada subpasta dentro de `dataset_unificado/`.

In [12]:
# Tamanho que toda imagem vai ter depois de carregada (altura, largura)
TAMANHO_IMAGEM = (300, 300)

# Quantidade de imagens processadas de cada vez (lote)
TAMANHO_LOTE = 32

# keras.utils.image_dataset_from_directory: função pronta do Keras que lê as imagens
# de uma pasta, redimensiona, e já separa em classes automaticamente.
dataset_treino = keras.utils.image_dataset_from_directory(
    PASTA_DADOS,
    image_size=TAMANHO_IMAGEM,
    batch_size=TAMANHO_LOTE,
    validation_split=0.2,   # reserva 20% dos dados
    subset="training",      # esta chamada pega os outros 80% (treino)
    seed=42                 # trava a divisão para ser sempre a mesma
)

dataset_teste = keras.utils.image_dataset_from_directory(
    PASTA_DADOS,
    image_size=TAMANHO_IMAGEM,
    batch_size=TAMANHO_LOTE,
    validation_split=0.2,
    subset="validation",    # esta chamada pega os 20% reservados (teste)
    seed=42                 # precisa ser a MESMA seed da chamada acima
)

# Lista com o nome de cada classe, na ordem que o Keras usa internamente (alfabética)
NOMES_CLASSES = dataset_treino.class_names
print("Classes encontradas:", NOMES_CLASSES)


Found 18781 files belonging to 4 classes.
Using 15025 files for training.


E0000 00:00:1789322958.733032    8597 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Found 18781 files belonging to 4 classes.
Using 3756 files for validation.
Classes encontradas: ['COVID-19', 'Normal', 'Pneumonia', 'Tuberculosis']


## Bloco 5 — Montar a rede neural (CNN)

Aqui montamos a arquitetura da rede, camada por camada, empilhando uma em cima da outra
com `modelo.add(...)`. A ordem importa: os dados passam pelas camadas na ordem em que
foram adicionadas.

**O padrão que se repete 5 vezes:**
1. `Conv2D` — procura padrões na imagem (bordas, texturas), usando vários filtros
2. `MaxPooling2D` — reduz o tamanho da imagem pela metade, mantendo só o mais importante

A cada repetição, aumentamos a quantidade de filtros (16, 32, 64, 128, 256), porque a rede
vai precisando reconhecer padrões cada vez mais complexos.

No final, achatamos tudo numa lista (`Flatten`) e usamos duas camadas (`Dense`) para
tomar a decisão final entre as 4 classes.

In [ ]:
# keras.Sequential: cria um "molde" vazio, onde vamos empilhar as camadas uma por uma
modelo = keras.Sequential()

# Rescaling: transforma os valores de pixel de 0-255 para 0.0-1.0 (deixa o treino mais estável)
# input_shape: formato de entrada esperado -> imagens 300x300 com 3 cores (RGB)
modelo.add(layers.Rescaling(1./255, input_shape=(300, 300, 3)))

# --- Bloco 1: Conv2D procura padrões, MaxPooling2D reduz o tamanho pela metade ---
# filters: quantidade de "detectores de padrão" diferentes nessa camada
# kernel_size: tamanho da janela que desliza sobre a imagem (3x3 pixels)
# activation='relu': zera qualquer valor negativo que sair da convolução
# padding='same': mantém o tamanho da imagem igual após a convolução
modelo.add(layers.Conv2D(filters=16, kernel_size=(3, 3), activation='relu', padding='same'))
modelo.add(layers.MaxPooling2D(pool_size=(2, 2)))

# --- Bloco 2 ---
modelo.add(layers.Conv2D(filters=32, kernel_size=(3, 3), activation='relu', padding='same'))
modelo.add(layers.MaxPooling2D(pool_size=(2, 2)))

# --- Bloco 3 ---
modelo.add(layers.Conv2D(filters=64, kernel_size=(3, 3), activation='relu', padding='same'))
modelo.add(layers.MaxPooling2D(pool_size=(2, 2)))

# --- Bloco 4 ---
modelo.add(layers.Conv2D(filters=128, kernel_size=(3, 3), activation='relu', padding='same'))
modelo.add(layers.MaxPooling2D(pool_size=(2, 2)))

# --- Bloco 5 ---
modelo.add(layers.Conv2D(filters=256, kernel_size=(3, 3), activation='relu', padding='same'))
modelo.add(layers.MaxPooling2D(pool_size=(2, 2)))

# Dropout: desliga 20% dos "neurônios" aleatoriamente durante o treino, para evitar
# que a rede decore demais as imagens de treino (overfitting)
modelo.add(layers.Dropout(0.2))

# Flatten: transforma o "bloco 3D" de números numa lista simples (1 dimensão só)
modelo.add(layers.Flatten())

# Dense: camada onde cada "neurônio" olha para TODOS os números de entrada
# 128 neurônios nessa camada intermediária
modelo.add(layers.Dense(128, activation='relu'))

# Última camada: 4 neurônios, um por classe.
# activation='softmax': transforma a saída em algo parecido com % de confiança por classe
modelo.add(layers.Dense(4, activation='softmax'))

# Mostra um resumo de todas as camadas, formatos e quantidade de parâmetros
modelo.summary()


## Bloco 6 — Configurar como a rede vai aprender

Antes de treinar, precisamos dizer ao modelo:
- qual algoritmo vai ajustar os pesos (`optimizer`)
- qual fórmula vai medir o quão errada está cada previsão (`loss`)
- qual métrica extra mostrar durante o treino, só para acompanharmos (`metrics`)

Isso não treina nada ainda — só prepara as regras que o treino vai seguir.

In [ ]:
# optimizer='adam': algoritmo que ajusta os pesos da rede a cada lote de imagens
# loss='sparse_categorical_crossentropy': fórmula padrão para classificação com várias classes
# metrics=['accuracy']: mostra a porcentagem de acerto durante o treino, para acompanharmos
modelo.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


## Bloco 7 — Treinar a rede

Aqui a rede efetivamente aprende: ela passa pelas imagens de treino várias vezes
(cada passada completa é uma "época"), ajustando os pesos a cada lote de 32 imagens.

Usamos um `ModelCheckpoint` para salvar automaticamente, em um arquivo no disco,
a versão do modelo que teve o melhor resultado no conjunto de teste — assim, mesmo
que as últimas épocas piorem (o que pode acontecer), ficamos com a melhor versão salva.

**Atenção:** este bloco pode demorar muitas horas, dependendo do computador.

In [ ]:
# ModelCheckpoint: "vigia" que roda durante o treino e salva o modelo em disco
# monitor="val_accuracy": observa a acurácia no conjunto de TESTE (não no treino)
# save_best_only=True: só salva um arquivo novo quando o resultado melhora
checkpoint = keras.callbacks.ModelCheckpoint(
    filepath="melhor_modelo.keras",
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

# modelo.fit: começa o treinamento de verdade
# epochs=30: passa pelas imagens de treino 30 vezes seguidas (como no artigo)
# validation_data: a cada época, testa também nas imagens de teste (sem aprender com elas)
# callbacks=[checkpoint]: liga o "vigia" que salva o melhor resultado
historico = modelo.fit(
    dataset_treino,
    validation_data=dataset_teste,
    epochs=30,
    callbacks=[checkpoint]
)


## Bloco 8 — Carregar a melhor versão salva

O treino pode ter terminado numa época que não foi a melhor (por causa de overfitting,
por exemplo). Por isso, carregamos do disco a versão que o checkpoint salvou como a melhor,
em vez de usar a versão que ficou na memória ao final do treino.

In [13]:
# keras.models.load_model: lê do disco um modelo já treinado e pronto para uso
melhor_modelo = keras.models.load_model("melhor_modelo.keras")


## Bloco 9 — Classificar as imagens de teste

Vamos passar todas as imagens de teste pelo modelo treinado, e guardar duas listas:
- a classe real de cada imagem (já sabíamos, pela pasta onde ela estava)
- a classe que o modelo previu para ela

Essas duas listas, lado a lado, são a base para calcular todas as métricas do próximo bloco.

In [ ]:
# Listas vazias que vamos preencher aos poucos
rotulos_reais = []
rotulos_previstos = []

# Percorre o dataset de teste, um lote de 32 imagens por vez
for lote_imagens, lote_rotulos in dataset_teste:

    # Pede ao modelo para classificar esse lote de imagens
    saida = melhor_modelo.predict(lote_imagens, verbose=0)

    # Para cada imagem, pega a classe com maior "confiança" (maior valor do softmax)
    previsao_do_lote = np.argmax(saida, axis=1)

    # Guarda os rótulos reais e as previsões desse lote nas listas finais
    rotulos_reais.extend(lote_rotulos.numpy())
    rotulos_previstos.extend(previsao_do_lote)

# Converte as listas para arrays numpy (formato que as métricas esperam)
rotulos_reais = np.array(rotulos_reais)
rotulos_previstos = np.array(rotulos_previstos)

print("Total de imagens avaliadas:", len(rotulos_reais))


W0000 00:00:1789322977.374128    8933 cpu_allocator_impl.cc:82] Allocation of 184320000 exceeds 10% of free system memory.
W0000 00:00:1789322977.715367    8927 cpu_allocator_impl.cc:82] Allocation of 184320000 exceeds 10% of free system memory.
W0000 00:00:1789322978.020582    8928 cpu_allocator_impl.cc:82] Allocation of 184320000 exceeds 10% of free system memory.
W0000 00:00:1789322978.334315    8925 cpu_allocator_impl.cc:82] Allocation of 184320000 exceeds 10% of free system memory.
W0000 00:00:1789322978.641186    8928 cpu_allocator_impl.cc:82] Allocation of 184320000 exceeds 10% of free system memory.


## Bloco 10 — Calcular as métricas por classe

Aqui usamos duas funções prontas do `scikit-learn`:

- `classification_report`: calcula precisão, recall e F1-score de cada classe.
- `confusion_matrix`: monta uma tabela mostrando, para cada classe real,
  o que o modelo previu — revela exatamente onde o modelo mais erra.

In [ ]:
# classification_report: recebe (respostas certas, previsões) e calcula as métricas prontas
relatorio = classification_report(
    rotulos_reais,
    rotulos_previstos,
    target_names=NOMES_CLASSES
)
print(relatorio)

# confusion_matrix: monta a tabela "real x previsto"
matriz = confusion_matrix(rotulos_reais, rotulos_previstos)

# Desenha a matriz de confusão como uma imagem colorida
plt.figure(figsize=(6, 5))
plt.imshow(matriz, cmap='Greens')
plt.colorbar()
plt.xticks(range(4), NOMES_CLASSES, rotation=45)
plt.yticks(range(4), NOMES_CLASSES)
plt.xlabel('Previsto pelo modelo')
plt.ylabel('Classe real')
plt.title('Matriz de Confusão')

# Escreve o número exato dentro de cada quadradinho da matriz
for linha in range(4):
    for coluna in range(4):
        plt.text(coluna, linha, matriz[linha, coluna], ha='center', va='center')

plt.tight_layout()
plt.show()


## Bloco 11 — Testar com uma amostra pequena (25 imagens por classe)

Além da avaliação geral (Bloco 10, que usa todas as imagens de teste de uma vez),
vamos testar o modelo numa amostra menor e específica: 25 imagens de cada classe,
olhando o resultado individual de cada uma — igual a um teste prático, em vez de
uma estatística com todas as imagens juntas.

Isso é feito em duas partes:
1. Separar 25 imagens de teste de cada classe.
2. Classificar essas 100 imagens e montar uma tabela mostrando, para cada classe
   real, quantas foram previstas corretamente e quantas foram confundidas com outra.

In [ ]:
# Um "balde" (lista) vazio para cada classe, onde vamos guardar até 25 imagens
amostras_por_classe = {0: [], 1: [], 2: [], 3: []}
QUANTIDADE_DESEJADA = 25

# Percorre o dataset de teste, lote por lote
for lote_imagens, lote_rotulos in dataset_teste:

    # zip: junta cada imagem com o rótulo correspondente, na mesma posição
    for imagem, rotulo in zip(lote_imagens.numpy(), lote_rotulos.numpy()):

        # Só guarda a imagem se aquele "balde" ainda não tiver 25 imagens
        if len(amostras_por_classe[rotulo]) < QUANTIDADE_DESEJADA:
            amostras_por_classe[rotulo].append(imagem)

    # Se todos os 4 "baldes" já tiverem 25 imagens, para de procurar
    baldes_completos = True
    for lista_de_imagens in amostras_por_classe.values():
        if len(lista_de_imagens) < QUANTIDADE_DESEJADA:
            baldes_completos = False
    if baldes_completos:
        break

# Confere quantas imagens foram coletadas em cada classe
for indice_classe, imagens in amostras_por_classe.items():
    print(NOMES_CLASSES[indice_classe] + ":", len(imagens), "imagens coletadas")


In [ ]:
# Dicionário para guardar os resultados: resultados["Normal"]["COVID-19"] = quantas
# imagens reais de Normal foram previstas (erradamente) como COVID-19
resultados = {}
for nome_real in NOMES_CLASSES:
    resultados[nome_real] = {}
    for nome_previsto in NOMES_CLASSES:
        resultados[nome_real][nome_previsto] = 0

# Para cada classe, classifica as 25 imagens coletadas
for indice_classe, imagens in amostras_por_classe.items():
    nome_classe_real = NOMES_CLASSES[indice_classe]

    imagens_array = np.array(imagens)
    saida = melhor_modelo.predict(imagens_array, verbose=0)

    # Pega a classe prevista para cada uma das 25 imagens
    previsoes_da_amostra = np.argmax(saida, axis=1)

    for previsao in previsoes_da_amostra:
        nome_classe_prevista = NOMES_CLASSES[previsao]
        resultados[nome_classe_real][nome_classe_prevista] += 1

# Imprime a tabela final, no mesmo formato da Tabela 5 do artigo
print(f"{'Real':<15} {'COVID-19':<10} {'Normal':<10} {'Pneumonia':<10} {'Tuberculosis':<12} {'Acertos'}")
for nome_real in NOMES_CLASSES:
    linha = resultados[nome_real]
    acertos = linha[nome_real]
    print(f"{nome_real:<15} {linha['COVID-19']:<10} {linha['Normal']:<10} {linha['Pneumonia']:<10} {linha['Tuberculosis']:<12} {acertos}/25")
